**Feature Selection Method:** ANOVA   
**Dataset:** QUT-DV25 (Opensnoop)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif

In [ ]:

# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()


In [ ]:
data.head()

,Package_Name,Total_Paths,Total_Error,Total_ File_Descriptor,Python_Related_Keywords,Install_Package_Keywords,Root_DIR_Installation,Temporary_DIR_Installation,Home_DIR_Installation,User_Access,Sys_Access,Etc_DIR_Installation,Other_DIR_Installation,Level
0,10Cent10-999.0.4.tar.gz,8045,8045,29,1083,2305,0,71,792,411,1225,151,5395,1
1,10Cent11-999.0.4.tar.gz,4790,4790,12,1616,843,2,87,1118,721,1200,104,1558,1
2,11Cent-999.0.0.tar.gz,26159,26159,27,3299,4536,0,200,2485,1072,1236,450,20716,1
3,11Cent-999.0.1.tar.gz,11194,11194,28,1521,2558,2,97,1035,891,1211,265,7693,1
4,11Cent-999.0.2.tar.gz,13561,13561,35,2656,5265,501,17,1497,1719,1219,291,8317,1


In [ ]:
data['Level'].unique()

array([1, 0], dtype=int64)

In [ ]:
all_pattern_columns = ['Total_Paths', 'Total_Error', 'Total_ File_Descriptor',
       'Python_Related_Keywords', 'Install_Package_Keywords',
       'Root_DIR_Installation', 'Temporary_DIR_Installation',
       'Home_DIR_Installation', 'User_Access', 'Sys_Access',
       'Etc_DIR_Installation', 'Other_DIR_Installation', 'Level']


data[all_pattern_columns] = data[all_pattern_columns].fillna(0)

# Separate target
X = data.drop(columns=['Level'])
y = data['Level']

if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

# Separate numerical and categorical features
numerical_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object','category']).columns.tolist()

X_num = X[numerical_cols]

# Encode categorical features as numeric codes
X_cat = X[categorical_cols].apply(lambda col: LabelEncoder().fit_transform(col.astype(str)))

# Combine numerical + encoded categorical features
X_encoded = pd.concat([X_num, X_cat], axis=1)

# Compute ANOVA F-scores
f_scores, _ = f_classif(X_encoded, y)

feature_df = pd.DataFrame({
    'feature': X_encoded.columns,
    'f_score': f_scores
})

# Select top 30% features by F-score
threshold = np.percentile(f_scores, 70)  # top 30%
feature_df['status'] = np.where(feature_df['f_score'] >= threshold, 'selected', 'deleted')

selected_features = feature_df[feature_df['status']=='selected']
deleted_features = feature_df[feature_df['status']=='deleted']

print("Selected features (name + F-score):\n", selected_features)
print("\nDeleted features (name + F-score):\n", deleted_features)


Selected features (name + F-score):
                     feature      f_score    status
3   Python_Related_Keywords  3530.834395  selected
4  Install_Package_Keywords  3042.642321  selected
7     Home_DIR_Installation  3717.820308  selected
8               User_Access  1925.052180  selected

Deleted features (name + F-score):
                        feature      f_score   status
0                  Total_Paths  1830.265007  deleted
1                  Total_Error  1830.265007  deleted
2       Total_ File_Descriptor    28.216872  deleted
5        Root_DIR_Installation  1356.632191  deleted
6   Temporary_DIR_Installation    67.277257  deleted
9                   Sys_Access     5.008980  deleted
10        Etc_DIR_Installation   831.338406  deleted
11      Other_DIR_Installation  1455.291634  deleted
12                Package_Name    22.678863  deleted
